# Sesgo inductivo

Otra de las técnicas arquitectońicas comúnmente utilizadas en la literatura es lo que se denomina **sesgo inductivo**. La idea es guiar u orientar al modelo a **priorizar** ciertas decisiones sobre otras, las cuales sabemos de antemano que tienen mayor potencial de entregar buenas soluciones. Tal como dice su nombre, es un **sesgo**, y por lo tanto se basa en suposiciones. Es una manera conveniente de facilitar el aprendizaje al modelo, pero a la vez, se debe tener cuidado de no confiar demasiado en este sesgo, de manera que el modelo tenga la libertad de aprender sus propios patrones (que podrían ir en una dirección totalmente distinta).

En el TSP, la **cercanía entre ciudades** es el sesgo utilizado por excelencia. Las ciudades más cercanas tienden a ser las siguientes en el tour. Esta es la base de algoritmos clásicos como *nearest neighbor* y la razón de que entreguen buenos resultados.

A nivel de arquitectura, existen dos maneras de aplicar este sesgo:

1. **Sesgo de salida**: Se aplica al final del modelo, justo antes de retornar los *logits*. Modifica directamente los *logits* sumándoles o restándoles el sesgo.

2. **Sesgo atencional**: Se aplica dentro del mecanismo de atención, modificando la ponderación/relevancia entre las *queries* y las *keys*.

Veremos en detalle cada uno de ellos a continuación.

## Preparación del entorno

Antes que nada, prepararemos los datos para el entrenamiento de los modelos.

Como vamos a aplicar sesgos basados en la **distancia** entre ciudades, se recomienda pasar directamente como entrada al modelo una **matriz de distancias**. Esto para evitar que el modelo tenga que recalcular internamente las distancias por cada iteración.

In [2]:
from instances.instances import generate_instances

instances = generate_instances(filename="TSP50.pkl", instance_count=1000, cities=50, seed=42)

from data.generation import generate_train_data
from data.adapters.input.distance import DistanceInputAdapter
from data.adapters.output.default import DefaultOutputAdapter

input_config = (DistanceInputAdapter, 50) # Añade la matriz de distancias como entrada al modelo
output_config = (DefaultOutputAdapter, 50)

generate_train_data(
    instance_file="TSP50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    size=5000
)

from data.preprocessing import load_dataset, split_dataset

# Split 80/20
train_file, val_file = split_dataset("train_data.h5", 4000)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 5000)
Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (4000 muestras) en: train_data_train.h5
Guardando Val puro (1000 muestras) en: train_data_val.h5
Dataset train_data_train.h5 cargado con 4000 muestras.
Dataset train_data_val.h5 cargado con 1000 muestras.


## Sesgo de salida (*output bias*)

El sesgo de salida se aplica directamente a los *logits* del modelo. Esto se representa mediante la siguiente ecuación:

$$\text{Output}_i = \mathbf{x}_i + \alpha \cdot \phi(i)$$

$\mathbf{x}_i$ es el *logit* asociado a la ciudad $i$. Este valor debe entenderse como un **puntaje**. Mientras más alto el puntaje, mayor es la probabilidad de que aquella ciudad sea la siguiente en la secuencia.

A este *logit* le sumamos un sesgo ponderado $\alpha \cdot \phi(i)$. Aquí $\alpha$ corresponde al ponderador, el cual generalmente se deja como parámetro aprendible. Delegamos al modelo la tarea de determinar qué tan relevante es este sesgo para la toma de decisiones. De esta manera, si el modelo asigna $\alpha = 0$, básicamente está ignorando el sesgo por completo, mientras que cualquier valor $\alpha > 0$ implica que el modelo está utilizando esta información para priorizar algunas decisiones sobre otras.

$\phi$ es la función que representa el sesgo, en este caso basado en la distancia. Una ciudad **más cercana** de la ciudad actual tiene más probabilidades de ser la indicada. Por el contrario, una ciudad **más distante** es menos probable de ser la siguiente en el tour. Por consiguiente, a mayor distancia, el puntaje (*logit*) debe ser menor.

Sea $c$ la ciudad actual y $i$ cualquiera de ciudades no visitadas, definimos nuestro sesgo $\phi$ como:

$$\phi(i) = -dist(c, i)$$

Una manera fácil de verlo es entender la distancia como una **penalización**. Si las ciudades están más cerca entre sí, la penalización será baja, respetando el puntaje original de salida. Si las ciudades están muy distantes, el puntaje se verá fuertemente reducido.

A continuación, entrenaremos un modelo que implementa este sesgo (revisar código en el archivo correspondiente).

In [ ]:
from models.output_bias import TSPTransformer

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
ob_model = TSPTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout,
)

from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

ob_model = sl_train(
    model=ob_model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    loss_fn=CrossEntropyLoss(),
    metrics=[Accuracy()],
    metrics_filename="metrics_output_bias.txt"
)

** Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 2.1300 | Val CrossEntropy: 1.4215
    Accuracy: 64.00%
Epoch 2/10
    Train CrossEntropy: 1.4040 | Val CrossEntropy: 1.1643
    Accuracy: 70.20%
Epoch 3/10
    Train CrossEntropy: 1.1210 | Val CrossEntropy: 0.8900
    Accuracy: 80.60%
Epoch 4/10
    Train CrossEntropy: 0.9445 | Val CrossEntropy: 0.8070
    Accuracy: 80.60%
Epoch 5/10
    Train CrossEntropy: 0.8734 | Val CrossEntropy: 0.7566
    Accuracy: 81.30%
Epoch 6/10
    Train CrossEntropy: 0.8189 | Val CrossEntropy: 0.7238
    Accuracy: 81.10%
Epoch 7/10
    Train CrossEntropy: 0.7796 | Val CrossEntropy: 0.7029
    Accuracy: 81.50%
Epoch 8/10
    Train CrossEntropy: 0.7590 | Val CrossEntropy: 0.6796
    Accuracy: 81.80%
Epoch 9/10
    Train CrossEntropy: 0.7419 | Val CrossEntropy: 0.6551
    Accuracy: 82.10%
Epoch 10/10
    Train CrossEntropy: 0.7198 | Val CrossEntropy: 0.6323
    Accuracy: 82.40%

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-

## Sesgo de atención (*attention bias*)

En esta variante, aplicamos el sesgo directamente en la **matriz de pesos** de atención del **encoder**.

Aquí, la idea es simple: dos ciudades están más correlacionadas en el grafo de ciudades mientras más cerca estén una de la otra. Este sesgo lo aplicamos convenientemente en el encoder, que es la parte donde el modelo representa la estructura global del problema. Al inyectar este sesgo, permitimos que el modelo sea capaz de visualizar, por ejemplo, *clústers* o agrupaciones de ciudades dentro del grafo. De esta manera, el modelo tiene la capacidad de priorizar ciertas regiones sobre otras a medida que va trazando el tour.

$$A_{ij} = \frac{\mathbf{q}_i \mathbf{k}_j^\top}{\sqrt{d_k}} + \alpha^{(l, h)} \cdot \phi(i, j)$$

La función $\phi$ modela la distancia entre las ciudades $i$ y $j$. Añadimos un signo negativo siguiendo la misma lógica que antes: a mayor distancia, mayor penalización al peso de atención:

$$\phi(i, j) = -dist(i, j)$$

Una diferencia clave es que aquí tendremos diferentes ponderadores $\alpha$ dependiendo de la capa $l$ y cabezal $h$ donde se esté aplicando la atención. Esto es importante, porque para cierta capa/cabezal, el modelo puede determinar que la distancia es sumamente relevante para modelar un determinado tipo de relación (por ejemplo, la pertenencia de dos ciudades a un mismo *cluster*), mientras que para otra combinación capa/cabezal, el modelo podría aprender sus propios patrones donde la distancia deja de ser un factor relevante.

Al igual que antes, entrenaremos un modelo que implementa el sesgo de atención.

In [6]:
from models.attention_bias import TSPTransformer

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
ab_model = TSPTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout,
)

from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

ab_model = sl_train(
    model=ab_model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    loss_fn=CrossEntropyLoss(),
    metrics=[Accuracy()],
    metrics_filename="metrics_att_bias.txt"
)

** Usando dispositivo: cpu



Epoch 1/10
    Train CrossEntropy: 2.3922 | Val CrossEntropy: 1.5371
    Accuracy: 61.60%
Epoch 2/10
    Train CrossEntropy: 1.4547 | Val CrossEntropy: 1.2191
    Accuracy: 70.60%
Epoch 3/10
    Train CrossEntropy: 1.1007 | Val CrossEntropy: 1.0436
    Accuracy: 75.30%
Epoch 4/10
    Train CrossEntropy: 0.9550 | Val CrossEntropy: 0.9246
    Accuracy: 77.70%
Epoch 5/10
    Train CrossEntropy: 0.8874 | Val CrossEntropy: 0.8965
    Accuracy: 78.10%
Epoch 6/10
    Train CrossEntropy: 0.8314 | Val CrossEntropy: 0.8639
    Accuracy: 78.50%
Epoch 7/10
    Train CrossEntropy: 0.8027 | Val CrossEntropy: 0.8107
    Accuracy: 79.50%
Epoch 8/10
    Train CrossEntropy: 0.7797 | Val CrossEntropy: 0.8039
    Accuracy: 77.70%
Epoch 9/10
    Train CrossEntropy: 0.7512 | Val CrossEntropy: 0.7741
    Accuracy: 79.10%
Epoch 10/10
    Train CrossEntropy: 0.7298 | Val CrossEntropy: 0.7560
    Accuracy: 80.70%

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-Framework/experiments/metri

# Validación

Por simplicidad, evaluamos cada variante por separado. Sin embargo, nada impide combinar ambas técnicas en una misma arquitectura, lo que tendría el potencial de obtener soluciones aún mejores.

In [13]:
from solvers.eval import evaluate
from data.adapters.input.distance import DistanceInputAdapter

# Configuración compartida
input_config = (DistanceInputAdapter, 50)
instance_file = "benchmarks/B50.pkl"

# Agrupamos los modelos en un diccionario para evaluar secuencialmente
models_to_evaluate = {
    "Sesgo de salida": ob_model,
    "Sesgo de atención": ab_model,
}

# Diccionario para almacenar las soluciones sin sobrescribirlas
evaluation_results = {}

for name, model in models_to_evaluate.items():
    print(f"\n{'='*50}")
    print(f"Evaluando: {name}")
    print(f"{'='*50}")
    
    model_sols, ort_sols = evaluate(
        model=model,
        instance_file=instance_file,
        input_adapter_config=input_config,
        num_workers=None  # Utiliza todos los núcleos de CPU disponibles
    )
    
    # Guardamos los resultados por si se necesitan analizar o graficar posteriormente
    evaluation_results[name] = {
        "model_sols": model_sols,
        "ort_sols": ort_sols
    }


Evaluando: Sesgo de salida
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.36
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      9.29% ± 6.01%


Evaluando: Sesgo de atención
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.51
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      11.90% ± 6.17%

